# 🔬 Module 18 — Statistiques inférentielles
> **Bootcamp Data Analyst — From Zero to Hero** | Niveau Intermédiaire · Module 18

---

## 🎯 Ce que tu seras capable de faire à la fin de ce module

- Expliquer la différence entre **décrire** un échantillon et **généraliser**/comparer avec confiance
- Formuler une hypothèse nulle (H0) et une hypothèse alternative (H1)
- Interpréter correctement une **p-value** — et éviter le piège d'interprétation le plus commun
- Comparer deux moyennes avec un **test t**, et lire un **intervalle de confiance**
- Tester une association entre deux variables catégorielles avec un **test du χ² (chi-carré)**
- Distinguer signification **statistique** et signification **pratique/métier**

> ⚠️ Ce module ne demande aucun calcul à la main — l'angle est **l'interprétation**. En tant que DA, ton rôle n'est pas de dériver les formules, mais de savoir lire un résultat de test et l'expliquer correctement à un∙e décideur∙se.

---

> ⏱️ **Durée estimée** : 2 heures
> 🛠️ **Outils** : Python (`scipy.stats`) · R (`t.test`, `chisq.test`)
> 📌 **Prérequis** : Module 02 et Module 11 (Niveau Débutant — statistiques descriptives)

---
## 1. Pourquoi aller au-delà de la description

Au Module 02 et au Module 11, tu as calculé des moyennes, médianes, écarts-types — tu **décrivais** un échantillon. Les statistiques inférentielles répondent à une question différente : *est-ce que ce que j'observe dans mon échantillon me permet de conclure quelque chose sur la population entière, ou sur une vraie différence entre deux groupes ?*

### Le scénario

Tu es Data Analyst chez **Bamba & Associés**. La DRH te pose une question sensible :

> *« On m'a signalé un possible écart de salaire entre les femmes et les hommes dans l'entreprise. Peux-tu vérifier si c'est un vrai problème structurel, ou juste du hasard sur nos effectifs actuels ? »*

Calculer "la moyenne des femmes est différente de la moyenne des hommes" ne suffit pas — avec un échantillon, il y aura **presque toujours** une petite différence, même si elle n'est due qu'au hasard de qui a été embauché. Les tests d'hypothèses permettent de répondre à la vraie question : *cet écart est-il assez marqué pour qu'on ne puisse pas l'attribuer au hasard ?*

---
## 2. Hypothèses et p-value

### H0 et H1

Un test d'hypothèse compare toujours deux affirmations :

- **H0 (hypothèse nulle)** : "il n'y a pas de différence/lien" — le scénario par défaut, celui qu'on cherche à réfuter
- **H1 (hypothèse alternative)** : "il y a une différence/lien"

Pour notre question RH : **H0** = "le salaire moyen est le même chez les femmes et les hommes" ; **H1** = "le salaire moyen diffère selon le genre".

### La p-value — ce qu'elle veut dire vraiment

La **p-value** est la probabilité d'observer un écart **au moins aussi marqué** que celui mesuré dans ton échantillon, **si H0 était vraie** (s'il n'y avait vraiment aucune différence dans la population).

- p-value **petite** (généralement < 0,05) → l'écart observé serait très improbable si H0 était vraie → on **rejette H0**, l'écart est jugé "statistiquement significatif"
- p-value **grande** (≥ 0,05) → l'écart observé est plausible même sous H0 → on ne peut **pas rejeter H0** — attention, ça ne veut **pas** dire "H0 est vraie" !

> ⚠️ **Le piège le plus commun** : la p-value **n'est pas** "la probabilité que H0 soit vraie", et ce n'est pas non plus "la probabilité de se tromper en rejetant H0". C'est la probabilité d'observer les données (ou plus extrême) **en supposant H0 vraie**. La nuance est subtile mais réelle — si tu dis un jour à ton manager *"il y a 3 % de chances que l'hypothèse nulle soit vraie"*, tu commets exactement cette erreur d'interprétation. La bonne formulation : *"si le salaire moyen était réellement identique entre les deux groupes, on aurait moins de 0,05 % de chances d'observer un écart aussi marqué que celui-ci sur cet échantillon"*.

---
## 3. Le test t — comparer deux moyennes

### Le jeu de données

On simule un extrait RH de 60 employés de Bamba & Associés (données fictives, générées pour l'exercice).

**Python :**
```python
import pandas as pd
import numpy as np

np.random.seed(7)

n = 60
genres = ["F"]*30 + ["H"]*30
departements = np.random.choice(["RH","Finance","Commercial","IT","Direction"], n, p=[0.2,0.2,0.25,0.2,0.15])
contrats = np.random.choice(["CDI","CDD","Stage"], n, p=[0.65,0.22,0.13])
villes = np.random.choice(["Abidjan","Bouaké","Yamoussoukro","San-Pédro"], n, p=[0.5,0.2,0.15,0.15])

dept_base = {"RH":400000,"Finance":500000,"Commercial":420000,"IT":480000,"Direction":900000}
salaire = []
for i in range(n):
    base = dept_base[departements[i]]
    ecart_genre = -85000 if genres[i] == "H" else 0
    bruit = np.random.normal(0, 55000)
    salaire.append(round(max(150000, base + ecart_genre + bruit) / 10000) * 10000)

df = pd.DataFrame({"genre": genres, "departement": departements, "contrat": contrats,
                    "ville": villes, "salaire": salaire})
```

### Comparer les deux groupes

```python
print(df.groupby("genre")["salaire"].agg(["mean", "std", "count"]).round(0))
```

Regarde l'écart entre les deux moyennes affichées. Est-ce significatif, ou juste le hasard de cet échantillon précis ? C'est la question à laquelle répond le test t.

**Python (`scipy.stats.ttest_ind`) :**
```python
from scipy import stats

f_salaire = df[df.genre == "F"]["salaire"]
h_salaire = df[df.genre == "H"]["salaire"]

resultat = stats.ttest_ind(f_salaire, h_salaire, equal_var=False)  # test t de Welch
print(f"t = {resultat.statistic:.3f}, p = {resultat.pvalue:.5f}")
```

**R (`t.test`, équivalent — Welch par défaut) :**
```r
resultat <- t.test(salaire ~ genre, data = df)
print(resultat)
```

### Interpréter ta p-value

Quelle que soit la valeur que tu obtiens, la lecture suit toujours la même logique :

- **p < 0,05** → tu rejettes H0. Formulation correcte : *"Si le salaire moyen était réellement identique entre femmes et hommes, la probabilité d'observer un écart aussi marqué que celui mesuré sur nos 60 employés serait de p × 100 %. L'écart observé est statistiquement significatif."*
- **p ≥ 0,05** → tu ne rejettes pas H0. Formulation correcte : *"On ne détecte pas d'écart statistiquement significatif entre les deux groupes sur cet échantillon."* (jamais *"il n'y a pas d'écart"* — voir section 2)

> 💡 **Pourquoi `equal_var=False` / le test de Welch ?** Le test t classique suppose que les deux groupes ont la même variance. En pratique, ce n'est presque jamais vrai. Le test de **Welch** ne fait pas cette hypothèse — c'est le choix par défaut et le plus sûr, dans les deux langages.

---
## 4. L'intervalle de confiance — aller au-delà du "significatif ou pas"

Une p-value te dit "significatif ou pas", mais pas **l'ampleur** de l'écart. L'**intervalle de confiance (IC)** complète l'information.

**Python :**
```python
ic = resultat.confidence_interval(confidence_level=0.95)
print(f"IC 95% : [{ic.low:.0f}, {ic.high:.0f}] FCFA")
```

**R :**
```r
resultat$conf.int
```

**Interprétation** : le résultat te donne une fourchette — *"on estime, avec 95 % de confiance, que l'écart réel de salaire moyen se situe entre [borne basse] et [borne haute] FCFA"*. C'est une information bien plus utile pour la DRH qu'un simple "c'est significatif" — elle donne une idée de l'**ampleur** du problème à corriger, pas juste de son existence.

---
## 5. Le test du χ² — tester un lien entre deux catégories

Deuxième question RH : *« Y a-t-il un lien entre la ville où travaille un∙e employé∙e et le type de contrat qu'il/elle a ? »* Ici, les deux variables sont **catégorielles** (pas des moyennes à comparer) — c'est le test du **χ² (chi-carré) d'indépendance** qu'il faut utiliser.

### Construire le tableau de contingence

**Python :**
```python
df["abidjan"] = df["ville"].apply(lambda v: "Abidjan" if v == "Abidjan" else "Autre ville")
df["type_contrat"] = df["contrat"].apply(lambda c: "CDI" if c == "CDI" else "Autre contrat")

table = pd.crosstab(df["abidjan"], df["type_contrat"])
print(table)
```

### Le test

```python
chi2, p_val, dof, attendus = stats.chi2_contingency(table)
print(f"chi2 = {chi2:.3f}, p = {p_val:.4f}, degrés de liberté = {dof}")
print("Effectifs attendus si aucun lien :")
print(attendus)
```

**R (`chisq.test`) :**
```r
table_r <- table(df$abidjan, df$type_contrat)
resultat_chi2 <- chisq.test(table_r)
print(resultat_chi2)
```

### Interpréter ta p-value

Même logique que pour le test t :

- **p < 0,05** → tu rejettes H0 : *"On détecte un lien statistiquement significatif entre la ville et le type de contrat."*
- **p ≥ 0,05** → tu ne rejettes pas H0 : *"On ne détecte pas de lien statistiquement significatif entre la ville et le type de contrat sur cet échantillon."*

> ⚠️ **Attention à la formulation** — ne dis jamais *"il n'y a pas de lien"* de façon catégorique. Un résultat non significatif peut vouloir dire deux choses très différentes : soit il n'y a vraiment aucun lien, soit l'échantillon est trop petit pour le détecter. Le test ne permet pas de trancher entre les deux.

### Une vérification avant de faire confiance au résultat

Le test du χ² n'est fiable que si les **effectifs attendus** (calculés sous H0, la variable `attendus` dans le code ci-dessus) sont suffisamment grands — la règle courante est **au moins 5 par case**. Vérifie ce point avant d'interpréter le résultat. Avec un tableau de contingence qui a des cases à 1 ou 2 individus attendus, le résultat du χ² devient peu fiable — il faut alors regrouper des catégories ou utiliser un test alternatif (test exact de Fisher).

---
## 6. Signification statistique ≠ importance pratique

Un dernier réflexe à avoir : un résultat peut être **statistiquement significatif** (p très petite) sans être **important en pratique**, et inversement.

- Avec un très grand échantillon (des millions de lignes), une différence minuscule et sans intérêt métier (ex : 500 FCFA d'écart de salaire) peut ressortir "statistiquement significative" simplement parce que l'échantillon est énorme.
- À l'inverse, un écart réel et important peut ne pas ressortir significatif si l'échantillon est trop petit — c'est le risque qu'on a signalé pour le test du χ² ci-dessus.

> 💡 **Réflexe DA** : toujours regarder la p-value **et** l'ampleur de l'effet (l'intervalle de confiance) **et** la taille de l'échantillon, avant de conclure quoi que ce soit à un∙e décideur∙se. La p-value seule ne raconte jamais toute l'histoire.

---
## ✅ Résumé du module

| Concept | Ce qu'il faut retenir |
|---|---|
| **Stats descriptives vs inférentielles** | Décrire un échantillon vs généraliser/comparer avec confiance |
| **H0 / H1** | H0 = "pas de différence/lien" (scénario par défaut) ; H1 = "il y a une différence/lien" |
| **p-value** | Probabilité d'observer un écart au moins aussi marqué **si H0 était vraie** — pas "la probabilité que H0 soit vraie" |
| **Seuil 0,05** | Convention courante : p < 0,05 → on rejette H0 ("statistiquement significatif") |
| **Test t (`ttest_ind` / `t.test`)** | Compare deux moyennes ; utiliser la version de Welch par défaut (`equal_var=False`) |
| **Intervalle de confiance** | Donne une fourchette plausible pour l'écart réel — complète la p-value |
| **Test du χ² (`chi2_contingency` / `chisq.test`)** | Teste un lien entre deux variables catégorielles, via un tableau de contingence |
| **Effectifs attendus ≥ 5** | Condition de validité du test du χ² — sinon regrouper les catégories ou utiliser un test exact |
| **"Non significatif" ≠ "pas de lien"** | Un résultat non significatif peut aussi venir d'un échantillon trop petit |
| **Significativité statistique ≠ importance pratique** | Toujours regarder p-value + ampleur de l'effet + taille d'échantillon ensemble |

---

## 🧠 Quiz — Vérifie ta compréhension

---

**Q1.** Que signifie une p-value de 0,03 pour un test comparant deux moyennes ?
- a) Il y a 3 % de chances que l'hypothèse nulle soit vraie
- b) Si l'hypothèse nulle était vraie (pas de vraie différence), on aurait 3 % de chances d'observer un écart au moins aussi marqué que celui mesuré
- c) L'écart mesuré a 97 % de chances d'être correct

<details>
<summary>👉 Voir la réponse</summary>

✅ **b)** — C'est la définition exacte de la p-value : une probabilité conditionnelle **sous H0**, pas une probabilité sur H0 elle-même. Les options a) et c) sont l'erreur d'interprétation la plus fréquente, à ne jamais reproduire devant un∙e décideur∙se.
</details>

---

**Q2.** Ton test t donne p = 0,45. Quelle formulation est correcte ?
- a) "On ne détecte pas d'écart statistiquement significatif entre les deux groupes sur cet échantillon"
- b) "Il n'y a aucune différence entre les deux groupes"
- c) "L'hypothèse nulle est vraie à 55 %"

<details>
<summary>👉 Voir la réponse</summary>

✅ **a)** — Un résultat non significatif ne prouve pas l'absence de différence (b) — elle peut exister mais ne pas être détectée (échantillon trop petit, bruit trop important). c) répète l'erreur d'interprétation de la p-value vue au Q1.
</details>

---

**Q3.** Pourquoi utiliser le test t de Welch (`equal_var=False`) plutôt que le test t classique par défaut ?
- a) Il est plus rapide à calculer
- b) Il ne suppose pas que les deux groupes ont la même variance — une hypothèse rarement vraie en pratique
- c) Il fonctionne uniquement avec R, pas Python

<details>
<summary>👉 Voir la réponse</summary>

✅ **b)** — Le test t classique (Student) suppose des variances égales entre les deux groupes. Le test de Welch relâche cette contrainte, ce qui le rend plus robuste et plus sûr par défaut, surtout quand rien ne garantit que les variances sont comparables (ce qui est le cas dans notre exemple).
</details>

---

**Q4.** Ton tableau de contingence pour un test du χ² a des cases avec seulement 1 ou 2 individus attendus. Que dois-tu faire ?
- a) Ignorer le problème, le test du χ² fonctionne toujours
- b) Regrouper des catégories pour augmenter les effectifs par case, ou utiliser un test exact de Fisher
- c) Multiplier chaque effectif par 5

<details>
<summary>👉 Voir la réponse</summary>

✅ **b)** — Le test du χ² n'est fiable que si les effectifs attendus sont suffisants (règle courante : ≥ 5 par case). En dessous, il faut regrouper des catégories ou utiliser une alternative comme le test exact de Fisher, plus adapté aux petits effectifs.
</details>

---

## ➡️ Module suivant

Tu sais maintenant tester une hypothèse et interpréter un résultat correctement. Dans le **Module 19**, on passe à l'automatisation Python — consommer une API et écrire un script réutilisable.

> **→ Module 19 — Automatisation Python (API & scripts)**